In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import GRU
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../Dataset/weekly_student_dataset.csv")

df.head()

,id_student,code_module,code_presentation,week,weekly_video_clicks,weekly_avg_quiz_score,weekly_assessments_completed,dropout
0,6516,AAA,2014J,-3,110,0.0,0.0,0
1,6516,AAA,2014J,-2,48,0.0,0.0,0
2,6516,AAA,2014J,-1,2,0.0,0.0,0
3,6516,AAA,2014J,0,96,0.0,0.0,0
4,6516,AAA,2014J,1,229,0.0,0.0,0


In [3]:
print(df.shape)

df.info()

df.head()

(627031, 8)
<class 'pandas.DataFrame'>
RangeIndex: 627031 entries, 0 to 627030
Data columns (total 8 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id_student                    627031 non-null  int64  
 1   code_module                   627031 non-null  str    
 2   code_presentation             627031 non-null  str    
 3   week                          627031 non-null  int64  
 4   weekly_video_clicks           627031 non-null  int64  
 5   weekly_avg_quiz_score         627031 non-null  float64
 6   weekly_assessments_completed  627031 non-null  float64
 7   dropout                       627031 non-null  int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 38.3 MB


,id_student,code_module,code_presentation,week,weekly_video_clicks,weekly_avg_quiz_score,weekly_assessments_completed,dropout
0,6516,AAA,2014J,-3,110,0.0,0.0,0
1,6516,AAA,2014J,-2,48,0.0,0.0,0
2,6516,AAA,2014J,-1,2,0.0,0.0,0
3,6516,AAA,2014J,0,96,0.0,0.0,0
4,6516,AAA,2014J,1,229,0.0,0.0,0


In [4]:
features = [

    "weekly_video_clicks",

    "weekly_avg_quiz_score",

    "weekly_assessments_completed"

]

X = df[features]

y = df["dropout"]

In [5]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

In [6]:
sequence_length = 4

X_sequences = []

y_sequences = []

students = df["id_student"].unique()

for student in students:

    student_data = df[
        df["id_student"] == student
    ].sort_values("week")

    values = student_data[features].values

    target = student_data["dropout"].values

    if len(values) >= sequence_length + 1:

        for i in range(

            len(values) - sequence_length

        ):

            X_sequences.append(

                values[i:i+sequence_length]

            )

            y_sequences.append(

                target[i+sequence_length]

            )

In [7]:
X_sequences = np.array(X_sequences)

y_sequences = np.array(y_sequences)

print(X_sequences.shape)

print(y_sequences.shape)

(526996, 4, 3)
(526996,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(

    X_sequences,

    y_sequences,

    test_size=0.2,

    random_state=42,

    stratify=y_sequences

)

In [9]:
model = Sequential()

model.add(

    LSTM(

        64,

        input_shape=(

            X_train.shape[1],

            X_train.shape[2]

        )

    )

)

model.add(

    Dropout(0.3)

)

model.add(

    Dense(

        32,

        activation="relu"

    )

)

model.add(

    Dense(

        1,

        activation="sigmoid"

    )

)

d:\nishanthini\Dropout prediction\project\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]

)

In [11]:
history = model.fit(

    X_train,

    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=32,

    verbose=1

)

Epoch 1/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 43s 4ms/step - accuracy: 0.9147 - loss: 0.2879 - val_accuracy: 0.9153 - val_loss: 0.2845
Epoch 2/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 53s 5ms/step - accuracy: 0.9150 - loss: 0.2849 - val_accuracy: 0.9153 - val_loss: 0.2842
Epoch 3/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 47s 4ms/step - accuracy: 0.9150 - loss: 0.2841 - val_accuracy: 0.9153 - val_loss: 0.2823
Epoch 4/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 57s 5ms/step - accuracy: 0.9150 - loss: 0.2837 - val_accuracy: 0.9153 - val_loss: 0.2818
Epoch 5/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 55s 5ms/step - accuracy: 0.9150 - loss: 0.2832 - val_accuracy: 0.9153 - val_loss: 0.2824
Epoch 6/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 52s 5ms/step - accuracy: 0.9150 - loss: 0.2828 - val_accuracy: 0.9153 - val_loss: 0.2814
Epoch 7/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 54s 5ms/step - accuracy: 0.9150 - loss: 0.2829 - val_accuracy: 0.9153 - val_loss: 0.2813
Epoch 8/20
10540/10540 ━━━━━━━━━━━━━━━━━━━━ 54s 5ms/step - accuracy: 

In [12]:
print(df["dropout"].value_counts())
print(df["dropout"].value_counts(normalize=True))

dropout
0    561107
1     65924
Name: count, dtype: int64
dropout
0    0.894863
1    0.105137
Name: proportion, dtype: float64


In [15]:
y_probability = model.predict(X_test)

y_prediction = (y_probability > 0.5).astype(int)

3294/3294 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step


In [16]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_prediction))
print("Precision :", precision_score(y_test, y_prediction))
print("Recall :", recall_score(y_test, y_prediction))
print("F1 Score :", f1_score(y_test, y_prediction))
print("ROC AUC :", roc_auc_score(y_test, y_probability))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_prediction))

print("\nClassification Report")
print(classification_report(y_test, y_prediction))

Accuracy : 0.9150569259962049
Precision : 0.5882352941176471
Recall : 0.0011165698972755694
F1 Score : 0.0022289089490694306
ROC AUC : 0.6389777789392217

Confusion Matrix
[[96437     7]
 [ 8946    10]]

Classification Report
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     96444
           1       0.59      0.00      0.00      8956

    accuracy                           0.92    105400
   macro avg       0.75      0.50      0.48    105400
weighted avg       0.89      0.92      0.87    105400



In [ ]:
loss, accuracy = model.evaluate(

    X_test,

    y_test,

    verbose=0

)

print("Testing Accuracy :", accuracy)

In [ ]:
y_probability = model.predict(X_test)

y_prediction = (

    y_probability > 0.5

).astype(int)

In [ ]:
print("Accuracy :", accuracy_score(y_test,y_prediction))

print("Precision :", precision_score(y_test,y_prediction))

print("Recall :", recall_score(y_test,y_prediction))

print("F1 Score :", f1_score(y_test,y_prediction))

print("ROC AUC :", roc_auc_score(y_test,y_probability))

print()

print(confusion_matrix(y_test,y_prediction))

print()

print(classification_report(y_test,y_prediction))

In [ ]:
future_risk = pd.DataFrame({

    "Actual":y_test,

    "Predicted":y_prediction.flatten(),

    "Risk Probability":y_probability.flatten()

})

In [ ]:
def classify_future_risk(prob):

    if prob < 0.30:

        return "Low"

    elif prob < 0.70:

        return "Medium"

    else:

        return "High"


future_risk["Future Risk"] = future_risk[
    "Risk Probability"
].apply(classify_future_risk)

future_risk.head()

In [ ]:
future_risk.to_csv(

    "../Dataset/future_risk_prediction.csv",

    index=False

)

print("Future Risk Prediction Dataset Saved Successfully")

In [ ]:
model = Sequential()

model.add(

    GRU(

        64,

        input_shape=(

            X_train.shape[1],

            X_train.shape[2]

        )

    )

)

model.add(Dropout(0.3))

model.add(Dense(32,activation="relu"))

model.add(Dense(1,activation="sigmoid"))